<a href="https://colab.research.google.com/github/brianiboh765-hub/crest-economic-forecaster/blob/main/eco_forecaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# 1. Sample UK quarterly inflation data (placeholder so you can test immediately)
dates = pd.date_range(start="2018-01-01", periods=28, freq="QE")
cpi = [
    2.5,
    2.4,
    2.4,
    2.1,
    1.9,
    2.0,
    1.7,
    1.3,
    1.5,
    0.6,
    0.5,
    0.6,
    0.7,
    2.5,
    3.1,
    5.4,
    7.0,
    9.4,
    10.1,
    10.5,
    10.1,
    7.9,
    6.7,
    4.0,
    3.2,
    2.0,
    1.7,
    2.5,
]
df = pd.DataFrame({"CPI": cpi}, index=dates)

# 2. Check stationarity (ADF Test)
adf_result = adfuller(df["CPI"])
print(f"ADF Statistic: {adf_result[0]:.4f} | p-value: {adf_result[1]:.4f}")

# 3. Train/Test split (keep last 4 quarters to test)
train = df["CPI"][:-4]
test = df["CPI"][-4:]

# 4. ARIMA forecast
arima_pred = ARIMA(train, order=(1, 1, 1)).fit().forecast(steps=4)
arima_rmse = np.sqrt(mean_squared_error(test, arima_pred))

# 5. Machine Learning (Random Forest with 3 lags)
df_ml = df.copy()
for i in range(1, 4):
  df_ml[f"lag_{i}"] = df_ml["CPI"].shift(i)
df_ml.dropna(inplace=True)

X_train, y_train = df_ml.drop("CPI", axis=1)[:-4], df_ml["CPI"][:-4]
X_test, y_test = df_ml.drop("CPI", axis=1)[-4:], df_ml["CPI"][-4:]

rf = RandomForestRegressor(random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print(f"ARIMA RMSE: {arima_rmse:.4f}")
print(f"Random Forest RMSE: {rf_rmse:.4f}")

ADF Statistic: -0.9781 | p-value: 0.7612
ARIMA RMSE: 2.5136
Random Forest RMSE: 2.4904


/tmp/ipykernel_961/3858974373.py:44: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_result = adfuller(df["CPI"])
